# 课后练习解答（06.02_experiment_overview）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 算术强度（FLOPs/Bytes）低于硬件 ridge point 的算子属于？
A. memory-bound
B. compute-bound
C. latency-bound
D. 无法判断

**解答：** A

**解析：** 算术强度低于平衡点说明访存时间主导，属于访存受限。


### 问题2（单选题）

**题目：** 小 batch 推理中，Conv+BN 融合收益主要来自？
A. 减少 kernel 启动与中间读写
B. 减少 FLOPs
C. 提高精度
D. 减少参数

**解答：** A

**解析：** 融合不改变计算量，但消除独立 BN kernel 与中间张量往返。


### 问题3（多选题）

**题目：** 合法且常见的算子优化手段包括？
A. 算子融合
B. 低精度
C. Tiling 分块
D. 增加中间张量

**解答：** ABC

**解析：** 增加中间张量会增大访存与显存，不是优化手段。


### 问题4（多选题）

**题目：** 判断优化方案是否有效需要结合？
A. 硬件峰值算力与带宽
B. 算术强度
C. batch_size 与 kernel 启动开销
D. 数据加载方式

**解答：** ABCD

**解析：** 性能是算子特征与硬件、调度、数据供给共同作用的结果。


### 问题5（判断题）

**题目：** 算术强度低于 ridge point 说明算子是 compute-bound。

**解答：** 错

**解析：** 低于平衡点说明访存时间占比更大，是 memory-bound。


### 问题6（判断题）

**题目：** 昇腾 CANN/ATC 在图编译阶段可能自动执行算子融合。

**解答：** 对

**解析：** ATC/CANN 图优化包含融合、常量折叠等步骤，可在编译期自动完成部分融合。


### 问题7（填空题）

**题目：** 硬件 ridge point 的计算公式为 ____。

**解答：** 峰值算力（FLOPs/s）÷ 峰值内存带宽（Bytes/s）


### 问题8（填空题）

**题目：** 对 memory-bound 算子，优化重点是减少 ____；对 compute-bound 算子，重点是提升 ____。

**解答：** 访存量；计算效率/并行度


### 问题9（简答题）

**题目：** 为什么基准测试需要 warmup、repeats 与同步？

**解答：** warmup 消除首次构图和缓存预热；repeats 抑制随机波动；同步保证 NPU 异步 kernel 完成后才计时，三者缺一会让结果失真。


### 问题10（简答题）

**题目：** 给定一个算子的算术强度，如何判断该用融合还是低精度？

**解答：** 将算术强度与硬件 ridge point 比较：memory-bound 优先减少访存量或融合相邻算子；compute-bound 优先降低 FLOPs（如低精度）或提高并行效率；实际还应结合精度要求做 A/B 测试。


### 问题11（代码设计题）

**题目：** 编写 conv2d_benchmark 函数，返回平均耗时与吞吐。

**解答：** ```python
def conv2d_benchmark(conv, x, warmup=10, repeats=50):
    for _ in range(warmup):
        conv(x)
    torch.npu.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        conv(x)
    torch.npu.synchronize()
    total = time.perf_counter() - start
    ms = total / repeats * 1000
    throughput = x.numel() / (total / repeats)
    return ms, throughput
```


### 问题12（单选题）

**题目：** 某算子算术强度 10 FLOPs/B，硬件 ridge point 20 FLOPs/B，它属于？
A. memory-bound
B. compute-bound
C. balanced
D. 无法确定

**解答：** A

**解析：** 强度低于平衡点，访存是瓶颈。


### 问题13（多选题）

**题目：** 算子融合的潜在收益包括？
A. 减少 kernel 启动
B. 减少中间张量读写
C. 降低峰值显存
D. 保证精度提升

**解答：** ABC

**解析：** 融合是等价变换，不保证精度变化。


### 问题14（判断题）

**题目：** 算子融合在任何 batch_size 下都会提升吞吐。

**解答：** 错

**解析：** 大 batch 下计算可能已是瓶颈，融合收益可能消失甚至因调度变差。


### 问题15（简答题）

**题目：** 设计一个 NPU 上的融合前后对比实验，写出变量控制与结论判断标准。

**解答：** 固定模型权重、输入 shape/dtype、batch_size、warmup/repeats 与同步方式，分别测量时延、吞吐、峰值显存和模块数量；正确性校验后再比较，若多次测量均值差异超过噪声范围才判定优化有效。
